# 그룹화 회귀와 더미 회귀

**🌐 언어:** [← English](/grouped-dummy-regression-en) | **한국어**

<small><em>작성자 권명석 · <a href="https://github.com/myeongseok-gwon">GitHub</a> · <a href="https://www.linkedin.com/in/myeongseok-gwon/?skipRedirect=true">LinkedIn</a></em></small>

그룹화 회귀는 집계된 데이터에서도 집단 평균과 집단 크기를 함께 보존하면 같은 핵심 아이디어를 복원할 수 있습니다. 더미 회귀는 이산 범주를 0/1 지표로 바꾸기 때문에, 계수는 평균 차이 또는 조건부 평균 차이로 해석됩니다.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.style.use("fivethirtyeight")


## 데이터 불러오기

`wage.csv`만 사용합니다.


In [2]:
wage = (pd.read_csv("data/wage.csv")
        .dropna(subset=["wage", "hours", "educ", "IQ", "lhwage"])
        .assign(hwage=lambda d: d["wage"] / d["hours"],
                T=lambda d: (d["educ"] > 12).astype(int))
        .copy())

wage[["wage", "hours", "hwage", "lhwage", "educ", "IQ", "T"]].head()


,wage,hours,hwage,lhwage,educ,IQ,T
0,769,40,19.225,2.956212,12,93,0
1,808,50,16.160,2.782539,18,119,1
2,825,40,20.625,3.026504,14,108,1
3,650,40,16.250,2.788093,12,96,0
4,562,40,14.050,2.642622,11,74,0


## 그룹화 회귀

임금을 교육연수별로 집계한 뒤, 가중치를 둔 그룹화 회귀와 가중치가 없는 그룹화 회귀를 개별자료 회귀와 비교합니다.


In [3]:
group_wage = (wage
              .groupby("educ", as_index=False)
              .agg(lhwage=("lhwage", "mean"),
                   count=("lhwage", "size")))

individual_model = smf.ols("lhwage ~ educ", data=wage).fit()
grouped_weighted_model = smf.wls("lhwage ~ educ", data=group_wage, weights=group_wage["count"]).fit()
grouped_unweighted_model = smf.ols("lhwage ~ educ", data=group_wage).fit()

pd.DataFrame(
    {
        "coef_on_educ": [
            individual_model.params["educ"],
            grouped_weighted_model.params["educ"],
            grouped_unweighted_model.params["educ"],
        ],
        "std_error": [
            individual_model.bse["educ"],
            grouped_weighted_model.bse["educ"],
            grouped_unweighted_model.bse["educ"],
        ],
    },
    index=["Individual OLS", "Grouped WLS", "Grouped OLS"],
).round(4)


,coef_on_educ,std_error
Individual OLS,0.0529,0.0065
Grouped WLS,0.0529,0.0057
Grouped OLS,0.0481,0.0059


In [ ]:
ax = group_wage.plot.scatter(x="educ", y="lhwage", s=group_wage["count"] * 2.5, alpha=0.65, figsize=(7, 4))
educ_grid = pd.DataFrame({"educ": sorted(wage["educ"].unique())})
ax.plot(educ_grid["educ"], grouped_weighted_model.predict(educ_grid), color="C1", label="Weighted grouped fit")
ax.plot(educ_grid["educ"], grouped_unweighted_model.predict(educ_grid), color="C2", label="Unweighted grouped fit")
ax.set_title("Grouped Regression Needs Weights")
ax.set_xlabel("Years of Education")
ax.set_ylabel("Log Hourly Wage")
ax.legend()
plt.show()


## 더미 회귀

교육연수가 12년을 초과하는지를 나타내는 처치 더미를 만든 뒤, 그 더미 모형과 교육연수를 완전한 범주형 변수로 둔 회귀를 비교합니다.


In [5]:
dummy_model = smf.ols("hwage ~ T", data=wage).fit()
conditional_dummy_model = smf.ols("hwage ~ T + IQ", data=wage).fit()
categorical_model = smf.ols("hwage ~ C(educ)", data=wage).fit()

pd.DataFrame(
    {
        "T_계수": [dummy_model.params["T"], conditional_dummy_model.params["T"]],
        "절편": [dummy_model.params["Intercept"], conditional_dummy_model.params["Intercept"]],
        "IQ_계수": [float("nan"), conditional_dummy_model.params["IQ"]],
    },
    index=["더미만", "더미 + IQ"],
).round(4)


,T_계수,절편,IQ_계수
더미만,4.9044,19.9405,NaN
더미 + IQ,3.1573,8.0956,0.1253


In [ ]:
ax = wage.plot.scatter(x="IQ", y="hwage", c="T", cmap="coolwarm", alpha=0.3, figsize=(7, 4))
plot_df = wage.sort_values(["T", "IQ"]).copy()
plot_df["y_hat"] = conditional_dummy_model.predict(plot_df)
for t_value, color, label in [(0, "C2", "T=0"), (1, "C1", "T=1")]:
    subset = plot_df.query("T == @t_value")
    ax.plot(subset["IQ"], subset["y_hat"], color=color, linewidth=2.5, label=label)
ax.set_title(f"Conditional Mean Difference = {conditional_dummy_model.params['T']:.2f}")
ax.set_xlabel("IQ")
ax.set_ylabel("Hourly Wage")
ax.legend()
plt.show()

wage.groupby("educ", as_index=False)["hwage"].mean().rename(columns={"hwage": "평균_시간당_임금"}).head()


## 결과

그룹화된 데이터에서는 큰 집단이 더 큰 영향을 받도록 가중치를 주면 개별자료 회귀 결과와 가까워집니다. 더미 회귀에서는 계수가 평균 차이로 해석되고, 범주형 명세를 쓰면 교육의 효과를 하나의 선형 기울기에 억지로 묶지 않아도 됩니다.


## 참고 자료

이 노트북은 Matheus Facure의 *Causal Inference for the Brave and True*를 주요 참고 자료로 작성되었습니다.

- **Facure, M. (2022)**. *Causal Inference for the Brave and True*. Online book.  
  [https://matheusfacure.github.io/python-causality-handbook/](https://matheusfacure.github.io/python-causality-handbook/)

- **Facure, M. (2022)**. 06 - Grouped and Dummy Regression. *Causal Inference for the Brave and True*.  
  [https://matheusfacure.github.io/python-causality-handbook/06-Grouped-and-Dummy-Regression.html](https://matheusfacure.github.io/python-causality-handbook/06-Grouped-and-Dummy-Regression.html)